In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)

mateo = pd.read_csv(r'C:\Users\jarun\OneDrive\Desktop\pm25-early-warning-cnx\data\raw\openmeteo_all_provinces_2023-.csv')
firms = pd.read_csv(r'C:\Users\jarun\OneDrive\Desktop\pm25-early-warning-cnx\data\processed\firms_daily_by_province.csv')

mateo.head()

,Datetime,PM25,Province,temperature_2m,relative_humidity_2m,precipitation,surface_pressure,wind_speed_10m,wind_direction_10m
0,2023-01-01 00:00:00,29.5,Chiang Mai,20.3,73,0.0,981.4,1.9,68
1,2023-01-01 01:00:00,29.6,Chiang Mai,19.0,80,0.0,981.1,3.3,41
2,2023-01-01 02:00:00,30.9,Chiang Mai,17.2,89,0.0,980.6,5.5,328
3,2023-01-01 03:00:00,32.0,Chiang Mai,18.7,80,0.0,980.4,4.2,340
4,2023-01-01 04:00:00,32.5,Chiang Mai,18.8,80,0.0,980.4,0.7,90


In [2]:
firms.head()

,date,Province,hotspot_count,frp_sum,frp_mean
0,2023-01-01,Chiang Mai,0.0,0.00,0.000
1,2023-01-01,Chiang Rai,4.0,24.78,6.195
2,2023-01-01,Lampang,2.0,2.67,1.335
3,2023-01-01,Lamphun,1.0,8.05,8.050
4,2023-01-01,Mae Hong Son,0.0,0.00,0.000


In [5]:
mateo['Datetime'] = pd.to_datetime(mateo['Datetime'])
mateo = mateo.sort_values(['Province', 'Datetime']).reset_index(drop=True)

firms['date'] = pd.to_datetime(firms['date'])
print(f'Open Mateo: {mateo.shape}')
print(f'FIRMS: {firms.shape}')

Open Mateo: (210432, 9)
FIRMS: (8768, 5)


In [6]:
# ── 2. Merge: join บน date + province 

# meteo เป็นรายชั่วโมง → ดึง date ออกมาเพื่อ join กับ hotspot รายวัน
mateo['date'] = mateo['Datetime'].dt.normalize()  # ดึง date ออกมา (normalize() จะ set เวลาเป็น 00:00:00)
df = mateo.merge(firms, on=['Province', 'date'], how='left')

# วันที่ไม่มีข้อมูล hotspot (เกินช่วง firms) → fill 0
df[['hotspot_count','frp_sum','frp_mean']] = (
    df[['hotspot_count','frp_sum','frp_mean']].fillna(0)
)

print(f'\nAfter merge: {df.shape}')
print(f'Null Check: \n {df.isnull().sum()[df.isnull().sum() > 0]}')


After merge: (210432, 13)
Null Check: 
 Series([], dtype: int64)


In [7]:
df.head()

,Datetime,PM25,Province,temperature_2m,relative_humidity_2m,precipitation,surface_pressure,wind_speed_10m,wind_direction_10m,date,hotspot_count,frp_sum,frp_mean
0,2023-01-01 00:00:00,29.5,Chiang Mai,20.3,73,0.0,981.4,1.9,68,2023-01-01,0.0,0.0,0.0
1,2023-01-01 01:00:00,29.6,Chiang Mai,19.0,80,0.0,981.1,3.3,41,2023-01-01,0.0,0.0,0.0
2,2023-01-01 02:00:00,30.9,Chiang Mai,17.2,89,0.0,980.6,5.5,328,2023-01-01,0.0,0.0,0.0
3,2023-01-01 03:00:00,32.0,Chiang Mai,18.7,80,0.0,980.4,4.2,340,2023-01-01,0.0,0.0,0.0
4,2023-01-01 04:00:00,32.5,Chiang Mai,18.8,80,0.0,980.4,0.7,90,2023-01-01,0.0,0.0,0.0


In [8]:
df.columns

Index(['Datetime', 'PM25', 'Province', 'temperature_2m',
       'relative_humidity_2m', 'precipitation', 'surface_pressure',
       'wind_speed_10m', 'wind_direction_10m', 'date', 'hotspot_count',
       'frp_sum', 'frp_mean'],
      dtype='str')

In [9]:
df.to_csv(r'C:\Users\jarun\OneDrive\Desktop\pm25-early-warning-cnx\data\processed\openmeteo_firms_merged.csv', index=False)